# 11 · Phase 10 — Model-free control: Monte Carlo and temporal-difference learning (24AIM304 Unit 3)

Each learner interacts with the MDP of Notebook 10 by **sampling**: one episode = draw a
training recording, choose the 5 components ε-greedily, receive that recording's reward.
One episode = one reward query, so 60,000 episodes use **≈ 1.4 %** of the queries that the
exhaustive DP solution needs.

| method | update target for Q(s_t, a_t) |
|---|---|
| on-policy first-visit **Monte Carlo** control | return G (the terminal reward) |
| **SARSA** | Q(s_{t+1}, a_{t+1}) |
| **Expected SARSA** | Σ_a π(a\|s_{t+1}) Q(s_{t+1}, a) |
| **Q-learning** | max_a Q(s_{t+1}, a) |
| **Double Q-learning** | Q_B(s_{t+1}, argmax_a Q_A(s_{t+1}, a)) — removes maximisation bias |
| **n-step SARSA** (n = 2, 3) | Q(s_{t+n}, a_{t+n}) (or G if the episode ends first) |

ε decays linearly 1.0 → 0.05 over 70 % of training; step size α_t = max(0.1, 1/N(s,a)).

In [ ]:
%run common/00_config.ipynb
%run common/01_signal_lib.ipynb
%run common/02_rl_lib.ipynb
import matplotlib.pyplot as plt

K = load_json(P["results"] / "chosen_K.json")["K"]
prob = RLProblem(CFG, P, K=K)
R = CFG["rl"]
dp_value = prob.mdp("train").policy_value(np.array(load_json(P["results"] / "policies" / "dp_optimum.json")["policy"]))
base_val = prob.evaluate(np.full(K, prob.space.baseline), "val")

## Correctness check on a synthetic MDP with a known optimum
Same tree (5 × 3 × 5 × 4 × 7), one context, true configuration values ~ N(0, 0.1²) plus
Gaussian per-recording noise. With low noise every learner must approach the DP optimum.

In [ ]:
rng_s = np.random.default_rng(0)
dims_s = prob.space.dims
true_v = rng_s.normal(0, 0.1, int(np.prod(dims_s)))
chk = []
for noise in (0.01, 0.3):
    Rs = (true_v[None] + rng_s.normal(0, noise, (2000, len(true_v)))).astype(np.float32)
    ctx0 = np.zeros(2000, int)
    ref = ConfigMDP(Rs, ctx0, 1, dims_s)
    best_v = ref.policy_value(value_iteration(ref)["policy"])
    for m in ("mc", "sarsa", "q_learning", "double_q", "dyna_q", "prioritized_sweeping"):
        out = run_tabular(ConfigMDP(Rs, ctx0, 1, dims_s, rng=np.random.default_rng(2)), m, 30000, eval_every=0)
        chk.append({"noise sd": noise, "method": m, "fraction of optimal value": ref.policy_value(out["policy"]) / best_v})
pd.DataFrame(chk).pivot(index="method", columns="noise sd", values="fraction of optimal value").round(3)

In [ ]:
def val_fn(pol):
    m = prob.evaluate(pol, "val")
    return {"val_reward": m["mean_reward"], "val_macro_f1": m["macro_f1"], "val_bal_acc": m["balanced_acc"]}


methods = {"mc": {}, "sarsa": {}, "expected_sarsa": {}, "q_learning": {}, "double_q": {}, "nstep_sarsa_2": {"n": 2}, "nstep_sarsa_3": {"n": 3}}
runs = {}
for name, hp in methods.items():
    algo = "nstep_sarsa" if name.startswith("nstep") else name
    for seed in R["seeds"]:
        t0 = time.time()
        out = run_tabular(prob.mdp("train", seed=seed), algo, R["episodes"], alpha=R["alpha"], eps_sched=R["epsilon"],
                          eval_every=2000, val_fn=val_fn, **hp)
        out["sec"] = time.time() - t0
        runs[(name, seed)] = out
    print(f"{name:15s} {np.mean([runs[(name, s)]['sec'] for s in R['seeds']]):.1f}s/run")
joblib_path = P["processed"] / "tabular_runs.joblib"
import joblib
joblib.dump({k: {"policy": v["policy"], "curve": v["curve"]} for k, v in runs.items()}, joblib_path)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(21, 4.5))
for name in methods:
    curves = [pd.DataFrame(runs[(name, s)]["curve"]) for s in R["seeds"]]
    ep = curves[0].episode
    for a, col in zip(ax, ["train_value", "val_reward", "val_macro_f1"]):
        m = np.mean([c[col] for c in curves], 0); sd = np.std([c[col] for c in curves], 0)
        a.plot(ep, m, label=name); a.fill_between(ep, m - sd, m + sd, alpha=0.15)
ax[0].axhline(dp_value, color="k", ls="--", label="DP optimum (exhaustive)"); ax[0].set_title("true training value of the greedy policy")
ax[1].axhline(0, color="gray", ls=":", label="fixed baseline"); ax[1].set_title("validation mean Δ-reward")
ax[2].axhline(base_val["macro_f1"], color="gray", ls=":", label="fixed baseline"); ax[2].set_title("validation macro-F1 (proxy)")
for a in ax:
    a.set_xlabel("episodes = reward queries"); a.legend(fontsize=7)
plt.tight_layout(); savefig(fig, "11_td_mc_learning_curves"); plt.show()

In [ ]:
rows = []
for (name, seed), out in runs.items():
    mv = prob.evaluate(out["policy"], "val")
    rows.append({"method": name, "seed": seed, "train_value": prob.mdp("train").policy_value(out["policy"]),
                 "fraction_of_DP_value": prob.mdp("train").policy_value(out["policy"]) / dp_value,
                 **{f"val_{k}": mv[k] for k in ("mean_reward", "macro_f1", "balanced_acc", "auroc")}, "sec": out["sec"]})
td_tab = pd.DataFrame(rows)
td_tab.to_csv(P["tables"] / "11_mc_td_results.csv", index=False)
td_tab.groupby("method", sort=False).agg(["mean", "std"]).drop(columns="seed").round(4)

## Budget sweep: do the bootstrapping methods catch up with more reward queries?

Rewards arrive only at the end of the 5-step episode and a single recording's reward is very
noisy, while there are 16,800 (context, configuration) leaves. Bootstrapped targets
(SARSA / Q-learning) are then built on leaf estimates from 1–3 samples; Q-learning's max
over them is optimistic, and the error propagates up the tree slowly. Monte Carlo uses the
actual return at every level at once, and n-step methods sit in between — the classic
bias–variance trade-off of n-step returns (Sutton & Barto, ch. 7).

In [ ]:
sweep = []
for episodes in (15000, 60000, 240000):
    for name in ("mc", "q_learning", "nstep_sarsa_3", "double_q"):
        algo = "nstep_sarsa" if name.startswith("nstep") else name
        hp = {"n": 3} if name.startswith("nstep") else {}
        out = run_tabular(prob.mdp("train", seed=0), algo, episodes, alpha=R["alpha"], eps_sched=R["epsilon"], eval_every=0, **hp)
        mv = prob.evaluate(out["policy"], "val")
        sweep.append({"episodes": episodes, "method": name, "fraction_of_DP_value": prob.mdp("train").policy_value(out["policy"]) / dp_value,
                      "val_mean_reward": mv["mean_reward"], "val_macro_f1": mv["macro_f1"]})
sweep = pd.DataFrame(sweep)
sweep.to_csv(P["tables"] / "11_budget_sweep.csv", index=False)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for name, g in sweep.groupby("method"):
    ax[0].plot(g.episodes, g.fraction_of_DP_value, "o-", label=name); ax[1].plot(g.episodes, g.val_mean_reward, "o-", label=name)
ax[0].set_xscale("log"); ax[1].set_xscale("log"); ax[0].set_ylabel("fraction of DP-optimal training value"); ax[1].set_ylabel("validation mean Δ-reward")
for a in ax:
    a.set_xlabel("episodes (log)"); a.legend()
plt.tight_layout(); savefig(fig, "11_budget_sweep"); plt.show()
sweep.pivot(index="episodes", columns="method", values="fraction_of_DP_value").round(3)